In [19]:
import pandas as pd
from google.colab import drive
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import statsmodels.api as sm
import warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score,TimeSeriesSplit
from sklearn.metrics import r2_score
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

df = pd.read_excel('/content/drive/MyDrive/colab đại học/môn kinh tế lượng tài chính/paper/data PLX.xlsx')
df.columns = df.columns.str.replace(' ', '_').str.lower()
# chia ra bộ event và bộ ko event để xây model
events =[
    '2022-02-24',  # Nga-Ukraina'
    '2023-04-03',  # OPEC+ cắt giảm sản lượng
    '2025-04-02',  # Trump nâng thuế
    '2026-03-02',  # Iran 2026 - bùng phát từ 28/02
]
# Lấy data cho từng event: từ ..... ngày trước đến ..... ngày trước tổng có .... tháng để train
N_TRAIN = 100  # số ngày giao dịch để train
BUFFER  = 10   # đệm lịch (ngày) trước event
WIN = 7    # cửa sổ test: ± ngày giao dịch quanh event
datasets = {}

for i, event_str in enumerate(events, 1):
    event_dt = pd.Timestamp(event_str)
    # TRAIN: lấy 60 ngày giao dịch, kết thúc trước buffer
    buffer_cutoff  = event_dt - pd.Timedelta(days=BUFFER)
    train_end_idx  = df[df['date'] <= buffer_cutoff].index.max()
    train_start_idx = train_end_idx - N_TRAIN + 1
    train_df = df.loc[train_start_idx:train_end_idx].copy()
    # TEST: ±5 ngày giao dịch quanh ngày event
    event_idx      = df[df['date'] >= event_dt].index.min()
    test_start_idx = max(0, event_idx - WIN+5) #nếu lấy 7 thì là chỉ 2 ngày trước sự kiện
    test_end_idx   = min(len(df) - 1, event_idx + WIN)
    test_df = df.loc[test_start_idx:test_end_idx].copy()

    datasets[f'event{i}_train'] = train_df
    datasets[f'event{i}_test']  = test_df

results = {}
for i in range(1, 5):
    train = datasets[f'event{i}_train']
    test = datasets[f'event{i}_test']
    x_train = train.drop(['date', 'plx'], axis=1)
    y_train = train['plx']
    x_test = test.drop(['date', 'plx'], axis=1)
    y_test = test['plx']
    model = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
    model.fit(x_train, y_train)

    y_pred_train = model.predict(x_train)
    print(f"Event {i} - R2 train: {r2_score(y_train, y_pred_train)}")

    y_pred = model.predict(x_test)
    test = test[['date', 'plx']].copy()
    test['predict'] = y_pred # thêm cột vào test
    test['AR'] = test['plx'] - test['predict']
    test['CAR'] = test['AR'].cumsum()
    print(test)
    t_ar, p_ar = stats.ttest_1samp(test['AR'], 0)
    t_car, p_car = stats.ttest_1samp(test['CAR'], 0)
    print(f"AR: p={p_ar} , CAR:p={p_car} ")
    print("-" * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Event 1 - R2 train: 0.9352154162349942
          date    plx    predict        AR        CAR
297 2022-02-22  54.42  50.655859  3.764141   3.764141
298 2022-02-23  54.86  50.896944  3.963056   7.727197
299 2022-02-24  55.65  50.507832  5.142168  12.869365
300 2022-02-25  54.77  50.580452  4.189548  17.058913
301 2022-02-28  54.68  50.548258  4.131742  21.190655
302 2022-03-01  53.80  50.548258  3.251742  24.442397
303 2022-03-02  53.98  50.515782  3.464218  27.906615
304 2022-03-03  55.12  50.740618  4.379382  32.285998
305 2022-03-04  53.63  50.740618  2.889382  35.175380
306 2022-03-07  55.56  50.119400  5.440600  40.615980
AR: p=5.858293284353228e-08 , CAR:p=0.0002453574368857371 
--------------------------------------------------
Event 2 - R2 train: 0.9168257197312877
          date    plx    predict        AR       CAR
584 2023-03-30  33.76  32.942739  0.